In [ ]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Silver Layer: Product Data Cleaning
# MAGIC 
# MAGIC **Purpose**: Transform raw product data from Bronze to Silver layer
# MAGIC 
# MAGIC **Schedule**: Daily at 6:00 AM
# MAGIC 
# MAGIC **Dependencies**: Bronze ingestion complete

# COMMAND ----------

# MAGIC %pip install -r /Workspace/Repos/etl-framework/requirements.txt

# COMMAND ----------

from pyspark.sql import SparkSession
from etl_framework.connectors.delta_lake_manager import DeltaLakeManager
from etl_framework.transformations.product_transforms import ProductTransformer
from etl_framework.quality.great_expectations_suites import ProductQualitySuite
from etl_framework.config.settings import get_settings
import logging

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# COMMAND ----------

# MAGIC %md
# MAGIC ## Initialize Configuration

# COMMAND ----------

settings = get_settings()
storage_path = settings.storage.abfss_root

# Initialize Delta Lake manager
delta_manager = DeltaLakeManager(spark, storage_path)
transformer = ProductTransformer()
quality_suite = ProductQualitySuite()

# COMMAND ----------

# MAGIC %md
# MAGIC ## Read from Bronze

# COMMAND ----------

bronze_df = spark.read \
    .format("delta") \
    .load(f"{storage_path}/bronze/products")

logger.info(f"Read {bronze_df.count()} records from Bronze")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Apply Transformations

# COMMAND ----------

silver_df = transformer.transform(bronze_df)

# Add Silver layer metadata
silver_df = silver_df \
    .withColumn("_silver_timestamp", current_timestamp()) \
    .withColumn("_transformation_version", lit("2.1.0"))

logger.info(f"Transformed to {silver_df.count()} Silver records")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Data Quality Validation

# COMMAND ----------

validation_results = quality_suite.validate(silver_df)

# Write results to monitoring table
spark.createDataFrame([validation_results]) \
    .write \
    .mode("append") \
    .saveAsTable("monitoring.data_quality_results")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Write to Silver with Optimization

# COMMAND ----------

delta_manager.write_silver(
    df=silver_df,
    table_name="products",
    merge_key="product_id",
    partition_cols=["category", "year"]
)

# Optimize table
delta_manager.optimize_table("products", zorder_cols=["product_id"])

# COMMAND ----------

# MAGIC %md
# MAGIC ## Update Lineage Metadata

# COMMAND ----------

spark.sql(f"""
INSERT INTO metadata.lineage_log (
    source_table,
    target_table,
    records_read,
    records_written,
    execution_time,
    notebook_path
) VALUES (
    'bronze.products',
    'silver.products',
    {bronze_df.count()},
    {silver_df.count()},
    current_timestamp(),
    '{dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()}'
)
""")